In [1]:
# Import libraries for preprocessing
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Create a folder for processed outputs
os.makedirs("outputs", exist_ok=True)

# Load the original dataset
df = pd.read_csv("Dataset_ATS_v2.csv")

# Display basic information
print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (7043, 10)


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


## Pre-Transformation Data Integrity Check

Before splitting or transforming the dataset, the original data is checked for missing values, blank strings, duplicate rows, invalid categories, impossible numerical values, outliers and logical inconsistencies. No values are changed during this inspection stage.

In [2]:
# Identify numerical and categorical columns
numerical_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges"
]

categorical_columns = [
    "gender",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "Contract",
    "Churn"
]

# 1. Missing and blank values
missing_counts = df.isnull().sum()

blank_counts = (
    df[categorical_columns]
    .apply(lambda column: column.str.strip().eq("").sum())
)

# 2. Fully duplicated rows
duplicate_count = df.duplicated().sum()

# 3. Expected categorical values
expected_categories = {
    "gender": {"Female", "Male"},
    "Dependents": {"No", "Yes"},
    "PhoneService": {"No", "Yes"},
    "MultipleLines": {"No", "Yes"},
    "InternetService": {"DSL", "Fiber optic"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "Churn": {"No", "Yes"}
}

invalid_category_counts = {}

for column, expected_values in expected_categories.items():
    actual_values = set(df[column].dropna().unique())
    invalid_values = actual_values - expected_values
    invalid_category_counts[column] = len(
        df[df[column].isin(invalid_values)]
    )

# 4. Impossible numerical values
invalid_senior_count = (~df["SeniorCitizen"].isin([0, 1])).sum()
invalid_tenure_count = ((df["tenure"] < 0) | (df["tenure"] > 72)).sum()
invalid_charges_count = (df["MonthlyCharges"] < 0).sum()

# 5. IQR-based outlier check
outlier_counts = {}

for column in ["tenure", "MonthlyCharges"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_counts[column] = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ).sum()

# 6. Logical consistency check
phone_line_conflict = (
    (df["PhoneService"] == "No") &
    (df["MultipleLines"] == "Yes")
).sum()

# Display inspection results
print("Missing values by column:")
print(missing_counts)

print("\nBlank values in categorical columns:")
print(blank_counts)

print("\nFully duplicated rows:", duplicate_count)

print("\nInvalid categorical values:")
print(pd.Series(invalid_category_counts))

print("\nImpossible numerical values:")
print("SeniorCitizen outside 0 or 1:", invalid_senior_count)
print("tenure outside 0 to 72:", invalid_tenure_count)
print("Negative MonthlyCharges:", invalid_charges_count)

print("\nIQR-based outliers:")
print(pd.Series(outlier_counts))

print("\nLogical inconsistencies:")
print(
    "PhoneService = No and MultipleLines = Yes:",
    phone_line_conflict
)

Missing values by column:
gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64

Blank values in categorical columns:
gender             0
Dependents         0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
Churn              0
dtype: int64

Fully duplicated rows: 302

Invalid categorical values:
gender             0
Dependents         0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
Churn              0
dtype: int64

Impossible numerical values:
SeniorCitizen outside 0 or 1: 0
tenure outside 0 to 72: 0
Negative MonthlyCharges: 0

IQR-based outliers:
tenure            0
MonthlyCharges    0
dtype: int64

Logical inconsistencies:
PhoneService = No and MultipleLines = Yes: 260


### Integrity Decisions

The 302 fully duplicated rows were retained because the dataset does not contain a unique customer identifier. Therefore, it is not possible to determine whether they represent duplicate records or different customers with identical characteristics. Removing them could result in valid customer records being lost.

No missing values, blank strings, invalid categories, impossible numerical values or IQR-based outliers were found.

A total of 260 records had `PhoneService = No` and `MultipleLines = Yes`. This combination is logically inconsistent because a customer cannot have multiple phone lines without phone service. These values were corrected to `MultipleLines = No` in a separate working copy. The original dataset was not overwritten.

In [3]:
# Create a clean working copy
df_clean = df.copy()

# Correct the logical inconsistency
logical_conflict_mask = (
    (df_clean["PhoneService"] == "No") &
    (df_clean["MultipleLines"] == "Yes")
)

corrected_row_count = logical_conflict_mask.sum()

df_clean.loc[
    logical_conflict_mask,
    "MultipleLines"
] = "No"

# Create an integrity decision report
integrity_report = pd.DataFrame({
    "Check": [
        "Missing values",
        "Blank strings",
        "Fully duplicated rows",
        "Invalid categorical values",
        "Impossible numerical values",
        "IQR-based outliers",
        "Phone-service logical conflicts"
    ],
    "Finding": [
        int(missing_counts.sum()),
        int(blank_counts.sum()),
        int(duplicate_count),
        int(sum(invalid_category_counts.values())),
        int(
            invalid_senior_count +
            invalid_tenure_count +
            invalid_charges_count
        ),
        int(sum(outlier_counts.values())),
        int(phone_line_conflict)
    ],
    "Action": [
        "No action required",
        "No action required",
        "Retained",
        "No action required",
        "No action required",
        "No action required",
        "Corrected MultipleLines to No"
    ],
    "Reason": [
        "No missing values were detected",
        "No blank strings were detected",
        "No unique customer ID is available",
        "All categories matched expected values",
        "All numerical values were within valid ranges",
        "No IQR-based outliers were detected",
        "Multiple lines cannot exist without phone service"
    ]
})

# Verify the correction
remaining_conflicts = (
    (df_clean["PhoneService"] == "No") &
    (df_clean["MultipleLines"] == "Yes")
).sum()

print("Rows corrected:", corrected_row_count)
print("Remaining logical conflicts:", remaining_conflicts)
print("Original dataset shape:", df.shape)
print("Clean dataset shape:", df_clean.shape)

display(integrity_report)

Rows corrected: 260
Remaining logical conflicts: 0
Original dataset shape: (7043, 10)
Clean dataset shape: (7043, 10)


,Check,Finding,Action,Reason
0,Missing values,0,No action required,No missing values were detected
1,Blank strings,0,No action required,No blank strings were detected
2,Fully duplicated rows,302,Retained,No unique customer ID is available
3,Invalid categorical values,0,No action required,All categories matched expected values
4,Impossible numerical values,0,No action required,All numerical values were within valid ranges
5,IQR-based outliers,0,No action required,No IQR-based outliers were detected
6,Phone-service logical conflicts,260,Corrected MultipleLines to No,Multiple lines cannot exist without phone service


## Stratified Training and Testing Split

The cleaned dataset is divided into training and testing sets before encoding and scaling. An 80:20 split is used because it provides enough data for model training while keeping a separate and sufficiently large test set for reliable evaluation. With 7,043 records, this produces approximately 5,634 training records and 1,409 testing records.

Stratification preserves the churn proportion in both sets, while `random_state=42` ensures that the same split can be reproduced whenever the code is run.

The target variable, `Churn`, is encoded separately as `No = 0` and `Yes = 1`. The preprocessing transformations will be fitted only on the training data and then applied to the testing data to prevent data leakage.


In [4]:
# Separate input features and target variable
X = df_clean.drop(columns=["Churn"])
y = df_clean["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Split the dataset using stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Compare target distributions
split_summary = pd.DataFrame({
    "Dataset": [
        "Full dataset",
        "Training set",
        "Testing set"
    ],
    "Rows": [
        len(y),
        len(y_train),
        len(y_test)
    ],
    "Non_Churn_Count": [
        int((y == 0).sum()),
        int((y_train == 0).sum()),
        int((y_test == 0).sum())
    ],
    "Churn_Count": [
        int((y == 1).sum()),
        int((y_train == 1).sum()),
        int((y_test == 1).sum())
    ],
    "Churn_Rate_Percent": [
        round(y.mean() * 100, 2),
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

display(split_summary)

X_train shape: (5634, 9)
X_test shape: (1409, 9)
y_train shape: (5634,)
y_test shape: (1409,)


,Dataset,Rows,Non_Churn_Count,Churn_Count,Churn_Rate_Percent
0,Full dataset,7043,5174,1869,26.54
1,Training set,5634,4139,1495,26.54
2,Testing set,1409,1035,374,26.54


## Categorical Encoding and Feature Scaling

Categorical predictors are transformed using one-hot encoding. The first category of each feature is dropped to avoid the dummy-variable trap, and previously unseen test categories are ignored safely.

The continuous numerical features, `tenure` and `MonthlyCharges`, are standardised using `StandardScaler`. `SeniorCitizen` is already represented as a binary 0/1 variable, so it is passed through without scaling.

The preprocessing transformer is fitted only on the training data. The fitted training parameters are then used to transform the testing data, preventing data leakage.

In [5]:
# Define feature groups
categorical_features = [
    "gender",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "Contract"
]

continuous_features = [
    "tenure",
    "MonthlyCharges"
]

binary_numeric_features = [
    "SeniorCitizen"
]

# Build the preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "continuous",
            StandardScaler(),
            continuous_features
        ),
        (
            "binary",
            "passthrough",
            binary_numeric_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Build the preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

# Fit only on training data
X_train_processed_array = (
    preprocessing_pipeline.fit_transform(X_train)
)

# Apply the fitted transformations to test data
X_test_processed_array = (
    preprocessing_pipeline.transform(X_test)
)

# Get the transformed feature names
processed_feature_names = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Convert transformed arrays into DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names,
    index=X_test.index
)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nProcessed feature names:")
print(X_train_processed.columns.tolist())

X_train_processed.head()

Processed training shape: (5634, 10)
Processed testing shape: (1409, 10)

Processed feature names:
['gender_Male', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'Contract_One year', 'Contract_Two year', 'tenure', 'MonthlyCharges', 'SeniorCitizen']


,gender_Male,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,Contract_One year,Contract_Two year,tenure,MonthlyCharges,SeniorCitizen
3757,1.0,1.0,1.0,0.0,0.0,0.0,1.0,-1.276682,-0.149881,0.0
3165,0.0,0.0,1.0,0.0,0.0,1.0,0.0,-0.096089,-0.315125,0.0
4912,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.555273,1.172071,1.0
3877,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.433143,0.048412,0.0
3818,1.0,0.0,1.0,1.0,1.0,0.0,1.0,-1.276682,-1.471834,0.0


## Post-Transformation Integrity and Consistency Validation

The processed training and testing datasets are validated after encoding and scaling. The checks confirm that row counts are preserved, feature structures are consistent, no missing or infinite values were introduced, encoded features contain only binary values, scaled training features have approximately zero mean and unit standard deviation, and the target remains binary.

Training and testing indices are also checked to confirm that no record appears in both sets.

In [6]:
# Identify encoded binary output columns
binary_output_columns = [
    column
    for column in X_train_processed.columns
    if column not in continuous_features
]

# Check binary values
train_binary_valid = all(
    set(X_train_processed[column].unique()).issubset({0, 1})
    for column in binary_output_columns
)

test_binary_valid = all(
    set(X_test_processed[column].unique()).issubset({0, 1})
    for column in binary_output_columns
)

# Check scaled training statistics
scaled_train_means = (
    X_train_processed[continuous_features]
    .mean()
    .round(6)
)

scaled_train_std = pd.Series(
    np.std(
        X_train_processed[continuous_features],
        axis=0,
        ddof=0
    ),
    index=continuous_features
).round(6)

# Check index overlap
overlapping_indices = len(
    set(X_train.index).intersection(set(X_test.index))
)

# Count missing and infinite values
train_missing = int(
    X_train_processed.isnull().sum().sum()
)

test_missing = int(
    X_test_processed.isnull().sum().sum()
)

train_infinite = int(
    np.isinf(X_train_processed.to_numpy()).sum()
)

test_infinite = int(
    np.isinf(X_test_processed.to_numpy()).sum()
)

# Create final validation report
validation_report = pd.DataFrame({
    "Validation_Check": [
        "Training row count preserved",
        "Testing row count preserved",
        "Train and test columns match",
        "Training missing values",
        "Testing missing values",
        "Training infinite values",
        "Testing infinite values",
        "Training encoded values are binary",
        "Testing encoded values are binary",
        "Target values are binary",
        "Train and test index overlap",
        "Scaled training means approximately zero",
        "Scaled training standard deviations approximately one"
    ],
    "Result": [
        len(X_train_processed) == len(X_train),
        len(X_test_processed) == len(X_test),
        list(X_train_processed.columns) ==
        list(X_test_processed.columns),
        train_missing,
        test_missing,
        train_infinite,
        test_infinite,
        train_binary_valid,
        test_binary_valid,
        set(y_train.unique()).issubset({0, 1}) and
        set(y_test.unique()).issubset({0, 1}),
        overlapping_indices,
        np.allclose(
            scaled_train_means.values,
            0,
            atol=0.000001
        ),
        np.allclose(
            scaled_train_std.values,
            1,
            atol=0.000001
        )
    ],
    "Expected": [
        True,
        True,
        True,
        0,
        0,
        0,
        0,
        True,
        True,
        True,
        0,
        True,
        True
    ]
})

validation_report["Status"] = np.where(
    validation_report["Result"] ==
    validation_report["Expected"],
    "PASS",
    "REVIEW"
)

print("Scaled training means:")
print(scaled_train_means)

print("\nScaled training standard deviations:")
print(scaled_train_std)

print("\nFinal validation report:")
display(validation_report)

Scaled training means:
tenure            0.0
MonthlyCharges    0.0
dtype: float64

Scaled training standard deviations:
tenure            1.0
MonthlyCharges    1.0
dtype: float64

Final validation report:


,Validation_Check,Result,Expected,Status
0,Training row count preserved,True,True,PASS
1,Testing row count preserved,True,True,PASS
2,Train and test columns match,True,True,PASS
3,Training missing values,0,0,PASS
4,Testing missing values,0,0,PASS
5,Training infinite values,0,0,PASS
6,Testing infinite values,0,0,PASS
7,Training encoded values are binary,True,True,PASS
8,Testing encoded values are binary,True,True,PASS
9,Target values are binary,True,True,PASS


## Final Preprocessing Summary

All post-transformation validation checks passed successfully. The categorical predictors were one-hot encoded with the first category dropped to prevent the dummy-variable trap. The continuous predictors were standardised using parameters fitted only on the training data.

The training and testing sets contain identical feature structures, no missing or infinite values, and no overlapping record indices. The target distribution was preserved through stratified splitting.

In [7]:
# Create a preprocessing summary
preprocessing_summary = pd.DataFrame({
    "Feature_Group": [
        "Categorical predictors",
        "Continuous predictors",
        "Existing binary predictor",
        "Target variable",
        "Dataset split"
    ],
    "Features": [
        ", ".join(categorical_features),
        ", ".join(continuous_features),
        ", ".join(binary_numeric_features),
        "Churn",
        "All records"
    ],
    "Method": [
        "OneHotEncoder(drop='first', handle_unknown='ignore')",
        "StandardScaler",
        "Passed through without scaling",
        "Mapped No=0 and Yes=1",
        "80:20 stratified split with random_state=42"
    ],
    "Fitted_On": [
        "Training data only",
        "Training data only",
        "Not applicable",
        "Applied before splitting",
        "Full cleaned dataset"
    ]
})

# Save the cleaned audit dataset
df_clean.to_csv(
    "outputs/cleaned_dataset.csv",
    index=False
)

# Save processed feature datasets
X_train_processed.to_csv(
    "outputs/X_train_processed.csv",
    index=True,
    index_label="record_index"
)

X_test_processed.to_csv(
    "outputs/X_test_processed.csv",
    index=True,
    index_label="record_index"
)

# Save target datasets
y_train.to_frame(name="Churn").to_csv(
    "outputs/y_train.csv",
    index=True,
    index_label="record_index"
)

y_test.to_frame(name="Churn").to_csv(
    "outputs/y_test.csv",
    index=True,
    index_label="record_index"
)

# Save reports
integrity_report.to_csv(
    "outputs/integrity_check_report.csv",
    index=False
)

validation_report.to_csv(
    "outputs/post_transformation_validation.csv",
    index=False
)

split_summary.to_csv(
    "outputs/train_test_split_summary.csv",
    index=False
)

preprocessing_summary.to_csv(
    "outputs/preprocessing_summary.csv",
    index=False
)

# Save the fitted preprocessing pipeline
import joblib

joblib.dump(
    preprocessing_pipeline,
    "outputs/preprocessing_pipeline.joblib"
)

# Verify all saved files
print("Saved preprocessing outputs:")

for file_name in sorted(os.listdir("outputs")):
    file_path = os.path.join("outputs", file_name)
    file_size_kb = os.path.getsize(file_path) / 1024

    print(f"- {file_name}: {file_size_kb:.1f} KB")

Saved preprocessing outputs:
- X_test_processed.csv: 104.5 KB
- X_train_processed.csv: 417.2 KB
- cleaned_dataset.csv: 327.1 KB
- integrity_check_report.csv: 0.6 KB
- post_transformation_validation.csv: 0.6 KB
- preprocessing_pipeline.joblib: 4.9 KB
- preprocessing_summary.csv: 0.5 KB
- train_test_split_summary.csv: 0.2 KB
- y_test.csv: 9.5 KB
- y_train.csv: 37.6 KB


In [8]:
# Compress all preprocessing outputs
import shutil

shutil.make_archive(
    "preprocessing_outputs",
    "zip",
    "outputs"
)

print("Created: preprocessing_outputs.zip")

Created: preprocessing_outputs.zip
